# 20｜ChemVista 一次性盲态外部验证

本笔记本只在19号完成并正式解锁后运行。模型、校准、筛查阈值和拒绝阈值全部来自 Development 阶段；ChemVista 标签仅在所有预测生成之后用于一次性终末评价，禁止据此调参或重选模型。

流程分三段：`env_ms` 冻结输入 → `env_dreams` 提取 DreaMS 表示 → `env_ms` 运行冻结模型并统计。各阶段有落盘检查点，意外中断后可复用。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path(r'E:\谱图预测PBT_终版')
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
ENV_DREAMS = Path(r'D:\Appinstallation\Anaconda\envs\env_dreams\python.exe')
OUT = ROOT / 'outputs/pbt_msms_v1/chemvista_external_20_v1'
SRC = ROOT / 'src'

RUN_PREPARE = True
RUN_DREAMS_EMBED = True
RUN_EVALUATION = True
DREAMS_BATCH_SIZE = 16
N_BOOTSTRAP = 1000

assert ENV_MS.is_file(), ENV_MS
assert ENV_DREAMS.is_file(), ENV_DREAMS
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
print('ROOT:', ROOT)
print('Notebook kernel:', sys.executable)
print('env_ms:', ENV_MS)
print('env_dreams:', ENV_DREAMS)

## 1. 解锁门控

必须显示 `READY_FOR_NOTEBOOK_20`、`external_test_unlocked = True`、`chemvista_used = False`。

In [ ]:
gate_path = ROOT / 'outputs/pbt_msms_v1/comparison_19_v1/19_chemvista_unlock_gate.json'
gate = json.loads(gate_path.read_text(encoding='utf-8'))
assert gate['status'] == 'READY_FOR_NOTEBOOK_20'
assert gate['external_test_unlocked'] is True
assert gate['chemvista_used'] is False
display(pd.Series(gate, name='value').to_frame())

## 2. 冻结 ChemVista 输入（env_ms）

该阶段固定1873张谱图、586个化合物以及 Transfer / selected Novel / natural Novel 三个队列，并生成所有模型共用的只读特征。

In [ ]:
prepare_manifest = OUT / '20_preparation_manifest.json'
if RUN_PREPARE or not prepare_manifest.exists():
    subprocess.run([str(ENV_MS), '-m', 'pbt_msms.chemvista_external', '--stage', 'prepare', '--root', str(ROOT)], check=True, cwd=ROOT, env=env)
prep = json.loads(prepare_manifest.read_text(encoding='utf-8'))
assert prep['status'] == 'PASS'
display(pd.Series({k: prep[k] for k in ['status', 'n_spectra', 'n_compounds', 'role_counts', 'ion_mode_counts']}, name='value').to_frame())

## 3. DreaMS 冻结表示（env_dreams，GPU）

此阶段自动调用独立的 `env_dreams`。默认16与7号已验收批次一致；不要修改输入谱图。

In [ ]:
embed_manifest = OUT / '20_dreams_embedding_manifest.json'
if RUN_DREAMS_EMBED and not embed_manifest.exists():
    subprocess.run([str(ENV_DREAMS), '-m', 'pbt_msms.chemvista_external', '--stage', 'embed', '--root', str(ROOT), '--batch-size', str(DREAMS_BATCH_SIZE)], check=True, cwd=ROOT, env=env)
embed = json.loads(embed_manifest.read_text(encoding='utf-8'))
assert embed['status'] == 'PASS'
display(pd.Series({k: embed[k] for k in ['status', 'shape', 'dtype', 'batch_size', 'gpu']}, name='value').to_frame())

## 4. 冻结模型一次性外部预测与评价（env_ms）

会运行8条直接候选管线以复现19号冻结的不确定性定义，并输出最终五个终点的 Transfer / Novel 指标、90%经验区间覆盖率、拒绝后指标和bootstrap置信区间。首次运行耗时主要来自90个 Transformer checkpoint 的外部推理；已完成的子结果会自动复用。

In [ ]:
run_manifest = OUT / '20_run_manifest.json'
if RUN_EVALUATION or not run_manifest.exists():
    subprocess.run([str(ENV_MS), '-m', 'pbt_msms.chemvista_external', '--stage', 'evaluate', '--root', str(ROOT), '--bootstrap', str(N_BOOTSTRAP)], check=True, cwd=ROOT, env=env)
run = json.loads(run_manifest.read_text(encoding='utf-8'))
qc = pd.read_csv(OUT / '20_qc_checks.csv')
metrics = pd.read_csv(OUT / '20_external_metrics.csv')
display(qc)
assert run['status'] == 'READY_FOR_NOTEBOOK_21', run
assert qc['passed'].astype(bool).all()
display(metrics[(metrics['cohort'].isin(['transfer', 'novel_all'])) & (~metrics['retained_only'])])

## 5. 主图与完成状态

看到全部QC通过和 `READY_FOR_NOTEBOOK_21` 才算20号完成。结果属于已知身份标准谱图的外部盲测，不是实际水样应用；下一步才进入葡萄牙污水案例。

In [ ]:
figure = OUT / 'figures/20_chemvista_transfer_vs_novel.png'
display(Image(filename=str(figure)))
print('STATUS:', run['status'])
print('最终逐化合物预测:', OUT / '20_final_selected_model_predictions.csv.gz')
print('外部指标:', OUT / '20_external_metrics.csv')
print('质控:', OUT / '20_qc_checks.csv')